# Lab 02b: IA em Escala (Modelo Híbrido: Spark + Scikit-Learn)

**Disciplina:** Sistemas Distribuídos para IA (1CXIA - S10)
**Ambiente:** Databricks Shared/Serverless
**Objetivo:** Processar volume massivo via Spark e treinar um modelo de fraude via Scikit-Learn no Driver.

---
## 🛠️ Estratégia Híbrida
1. **Spark:** Heavy lifting de dados (1 Milhão de linhas).
2. **Pandas/Scikit-Learn:** Treinamento no Driver (Reduzido para 100k linhas para estabilidade do gRPC).
3. **Pandas UDF:** Predição em escala (Distribuindo o modelo Python).

In [ ]:
from pyspark.sql.functions import col, when, rand, lit

# 1. Gerando dados
df_spark = spark.range(1000000).withColumn("valor", (rand() * 1000)) \
    .withColumn("tipo_index", when(rand() > 0.5, 1.0).otherwise(0.0)) \
    .withColumn("label", when(rand() > 0.9, 1.0).otherwise(0.0))

# 2. ETL Distribuído
df_clean = df_spark.filter(col("valor") > 0).select("valor", "tipo_index", "label")
print(f"Volume total no Spark: {df_clean.count()}")

## 📥 Passo 2: Conversão Segura (gRPC Limit)
No Databricks Serverless, o limite de tráfego para o Driver é **128MB**. 
Limitaremos a conversão para **100.000 linhas** (~60MB) para garantir o sucesso do treinamento local.

In [ ]:
# Trazendo amostra significativa para o Driver
pdf = df_clean.limit(100000).toPandas() 

print(f"Dados carregados no Driver: {len(pdf)} linhas.")
pdf.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = pdf[["valor", "tipo_index"]]
y = pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=50, n_jobs=-1)
model.fit(X_train, y_train)
print("Modelo treinado no Driver!")

---
## ✅ Gabarito: Predição Distribuída (Pandas UDF)
Para prever em milhões de linhas sem estourar o gRPC, o segredo é **não trazer o dado de volta**, apenas processar e visualizar uma amostra final.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

@pandas_udf(DoubleType())
def predict_fraud_udf(valor: pd.Series, tipo_index: pd.Series) -> pd.Series:
    X_batch = pd.DataFrame({"valor": valor, "tipo_index": tipo_index})
    return pd.Series(model.predict(X_batch))

# Aplicando em uma amostra leve para o display (Evitando RESOURCE_EXHAUSTED)
df_final = df_spark.limit(1000).withColumn("predicao", predict_fraud_udf(col("valor"), col("tipo_index")))
display(df_final.select("valor", "tipo_index", "predicao"))